# Debug: EIA Natural Gas Storage (`fetch_eia.py`)

Step-by-step debug notebook for the EIA API v2 storage pipeline.

**Pipeline:** `pipeline/fetch_eia.py`  
**Output:** `data/eia_storage.parquet`  
**Source:** EIA API v2 — requires `EIA_API_KEY`

---
Set your API key in a `.env` file or export `EIA_API_KEY` before running.  
Register for a free key at https://www.eia.gov/opendata/register.php

## 1. Imports & Dependencies

In [ ]:
import importlib
import os
import sys
from datetime import date, timedelta
from pathlib import Path

required = ["pandas", "requests", "pyarrow", "dotenv"]
missing = []
for pkg in required:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if missing:
    print(f"FAIL  Missing packages: {missing}")
else:
    import pandas as pd
    import requests
    from dotenv import load_dotenv
    load_dotenv()
    print(f"PASS  All packages available — pandas {pd.__version__}")

## 2. API Key Check

In [ ]:
API_KEY = os.environ.get("EIA_API_KEY", "")

if not API_KEY:
    print("FAIL  EIA_API_KEY is not set")
    print("      Options:")
    print("        1. Create a .env file in the project root with: EIA_API_KEY=your_key_here")
    print("        2. Export in shell: export EIA_API_KEY=your_key_here")
    print("        3. Register free at https://www.eia.gov/opendata/register.php")
else:
    masked = API_KEY[:4] + "*" * (len(API_KEY) - 4)
    print(f"PASS  EIA_API_KEY found: {masked}")

## 3. Configuration

In [ ]:
EIA_BASE_URL = "https://api.eia.gov/v2"
STORAGE_SERIES = "NG.NW2_EPG0_SWO_R48_BCF.W"
ROLLING_YEARS = 5
OUTPUT_PATH = Path("../data/eia_storage.parquet")

end_date = date.today()
# Fetch extra year for 5yr-average computation
start_date = end_date - timedelta(days=(ROLLING_YEARS + 1) * 365)

print(f"Series     : {STORAGE_SERIES}")
print(f"Date range : {start_date} → {end_date}")
print(f"Output     : {OUTPUT_PATH.resolve()}")
print(f"Exists     : {OUTPUT_PATH.exists()}")

## 4. Raw API Request — Inspect URL & Response

In [ ]:
if not API_KEY:
    print("SKIP  No API key — set EIA_API_KEY and re-run")
else:
    url = f"{EIA_BASE_URL}/seriesid/{STORAGE_SERIES}"
    params = {
        "api_key": API_KEY,
        "frequency": "weekly",
        "data[0]": "value",
        "start": start_date.isoformat(),
        "end": end_date.isoformat(),
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "offset": 0,
        "length": 5000,
    }

    # Show the effective URL (mask key)
    display_params = {k: ("***" if k == "api_key" else v) for k, v in params.items()}
    print(f"URL    : {url}")
    print(f"Params :")
    for k, v in display_params.items():
        print(f"  {k}: {v}")
    print()

    try:
        resp = requests.get(url, params=params, timeout=30)
        print(f"Status : {resp.status_code}")

        if resp.status_code == 403:
            print("FAIL  403 Forbidden — API key is invalid or expired")
            print(f"      Response: {resp.text[:300]}")
        elif resp.status_code == 404:
            print("FAIL  404 Not Found — series ID may be wrong")
            print(f"      Series: {STORAGE_SERIES}")
        elif resp.status_code != 200:
            print(f"FAIL  Unexpected status {resp.status_code}")
            print(f"      Response: {resp.text[:500]}")
        else:
            payload = resp.json()
            response_block = payload.get("response", {})
            rows = response_block.get("data", [])
            total = response_block.get("total", "(not provided)")
            print(f"PASS  Got {len(rows)} rows (API reports total={total})")
            print()
            print("First row:", rows[0] if rows else "(empty)")
            print("Last row: ", rows[-1] if rows else "(empty)")
    except requests.Timeout:
        print("FAIL  Request timed out after 30s — check network connectivity")
    except Exception as e:
        print(f"FAIL  Exception: {e}")

## 5. Parse Response into DataFrame

In [ ]:
if not API_KEY:
    print("SKIP  No API key")
elif resp.status_code != 200:
    print("SKIP  API request failed — fix errors above first")
else:
    rows = payload.get("response", {}).get("data", [])

    storage = pd.DataFrame(rows)
    print(f"Raw columns: {list(storage.columns)}")
    print()

    storage = storage.rename(columns={"period": "date", "value": "working_gas_bcf"})
    storage["date"] = pd.to_datetime(storage["date"])
    storage["working_gas_bcf"] = pd.to_numeric(storage["working_gas_bcf"], errors="coerce")
    storage = storage[["date", "working_gas_bcf"]].sort_values("date").reset_index(drop=True)

    print(f"Shape : {storage.shape}")
    print(f"Dtypes: {storage.dtypes.to_dict()}")
    print()
    print("First 3 rows:")
    print(storage.head(3).to_string())
    print()
    print("Last 3 rows:")
    print(storage.tail(3).to_string())

    nan_count = storage["working_gas_bcf"].isna().sum()
    if nan_count > 0:
        print(f"\nWARN  {nan_count} NaN values in working_gas_bcf")
    else:
        print(f"\nPASS  No NaN values in working_gas_bcf")

## 6. Compute Net Change (week-over-week)

In [ ]:
if not API_KEY or resp.status_code != 200:
    print("SKIP")
else:
    storage["net_change_bcf"] = storage["working_gas_bcf"].diff()

    print("Net change stats:")
    print(storage["net_change_bcf"].describe().round(1).to_string())
    print()

    # Sanity check: typical weekly change is -150 to +150 Bcf
    extreme = storage[storage["net_change_bcf"].abs() > 200]
    if not extreme.empty:
        print(f"WARN  {len(extreme)} rows with |net_change| > 200 Bcf (may indicate data gaps):")
        print(extreme[["date", "working_gas_bcf", "net_change_bcf"]].to_string())
    else:
        print("PASS  No extreme weekly changes detected")

## 7. Compute 5-Year Rolling Average

For each week of year, average working gas across the prior 5 calendar years.

In [ ]:
if not API_KEY or resp.status_code != 200:
    print("SKIP")
else:
    def compute_5yr_average(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["week"] = df["date"].dt.isocalendar().week.astype(int)
        df["year"] = df["date"].dt.year

        def _avg(row):
            mask = (
                (df["week"] == row["week"]) &
                (df["year"].between(row["year"] - 5, row["year"] - 1))
            )
            return df.loc[mask, "working_gas_bcf"].mean()

        df["value_5yr_avg"] = df.apply(_avg, axis=1)
        return df[["date", "value_5yr_avg"]]

    print("Computing 5-year averages (this may take ~10 seconds) ...")
    avg_df = compute_5yr_average(storage)
    storage = storage.merge(avg_df, on="date", how="left")

    # How many rows have a valid 5yr avg?
    has_avg = storage["value_5yr_avg"].notna().sum()
    total = len(storage)
    print(f"PASS  5yr avg computed — valid for {has_avg}/{total} rows")

    # Rows without an average (expected for the first ~5 years of history)
    no_avg = storage[storage["value_5yr_avg"].isna()]
    if not no_avg.empty:
        print(f"      {len(no_avg)} rows without avg (oldest history, expected):")
        print(f"      {no_avg['date'].min().date()} → {no_avg['date'].max().date()}")

    print()
    print("Latest 5 rows:")
    print(storage.tail(5)[["date", "working_gas_bcf", "net_change_bcf", "value_5yr_avg"]].to_string())

## 8. Trim to Rolling Window & Final Checks

In [ ]:
if not API_KEY or resp.status_code != 200:
    print("SKIP")
else:
    cutoff = pd.Timestamp(date.today() - timedelta(days=ROLLING_YEARS * 365))
    final = storage[storage["date"] >= cutoff].reset_index(drop=True)

    print(f"After trim: {len(final)} rows (cutoff: {cutoff.date()})")
    print(f"Date range: {final['date'].min().date()} → {final['date'].max().date()}")
    print()

    issues = []

    # Expected ~260 weekly observations per 5 years
    if len(final) < 200:
        issues.append(f"Only {len(final)} rows — expected ~260 for 5-year weekly data")

    # Gap detection: weeks between consecutive dates
    gaps = final["date"].diff().dt.days
    large_gaps = gaps[gaps > 14]  # more than 2 weeks
    if not large_gaps.empty:
        issues.append(f"{len(large_gaps)} gaps > 14 days between observations")
        print("Gaps detected at:")
        for idx in large_gaps.index:
            print(f"  {final.loc[idx-1, 'date'].date()} → {final.loc[idx, 'date'].date()} ({gaps[idx]:.0f} days)")

    if issues:
        print("WARN  Issues:")
        for i in issues:
            print(f"  - {i}")
    else:
        print("PASS  Final dataset looks healthy")

## 9. Compare to Saved Parquet

In [ ]:
if OUTPUT_PATH.exists():
    saved = pd.read_parquet(OUTPUT_PATH)
    print(f"Saved parquet: {len(saved)} rows")
    print(f"Columns      : {list(saved.columns)}")
    print(f"Date range   : {saved['date'].min().date()} → {saved['date'].max().date()}")
    print()
    print("Latest 3 saved rows:")
    print(saved.tail(3).to_string())

    if API_KEY and resp.status_code == 200:
        fresh_max = final["date"].max()
        saved_max = saved["date"].max()
        lag = (fresh_max - saved_max).days
        if lag > 10:
            print(f"\nWARN  Saved parquet is {lag} days behind fresh fetch")
        else:
            print(f"\nPASS  Saved parquet is up to date (lag = {lag} days)")
else:
    print(f"INFO  No saved parquet at {OUTPUT_PATH} — run pipeline to generate it")

## 10. Write Output (optional)

In [ ]:
# if API_KEY and resp.status_code == 200:
#     OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#     final.to_parquet(OUTPUT_PATH, index=False)
#     print(f"Wrote {len(final)} rows → {OUTPUT_PATH}")